In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
import time
import os
import pandas as pd

In [20]:
def set_up_driver():
    chrome_options = Options()
    #chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [21]:
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

def scroll_to_load_all(driver, wait):

    last_count = 0
    same_count = 0
    
    while same_count < 3:  
        body = driver.find_element(By.TAG_NAME, 'body')
        for _ in range(5):
            body.send_keys(Keys.PAGE_DOWN)
            time.sleep(1)
        time.sleep(2)  

        current_count = len(driver.find_elements(By.CSS_SELECTOR, "a[href*='/thuoc'], a[href*='/dau']"))
        if current_count == last_count:
            same_count += 1
        else:
            same_count = 0
            last_count = current_count
        print(f"Đã load: {last_count} sản phẩm", end='\r')

    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.HOME)
    print(f"Hoàn tất load: {last_count} sản phẩm")

In [22]:
def close_popups(driver, wait):
    try:
        pop_up = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "p-5")))
        close_menu = pop_up.find_element(By.XPATH, "/html/body/div[2]/div/div[2]/div/div/div/div[3]")
        pop_up_close = close_menu.find_element(By.XPATH, "/html/body/div[2]/div/div[2]/div/div/div/div[3]/button[2]")
        pop_up_close.click()  
    except Exception as e:
        print(f"Lỗi {e}")

In [23]:
def category_menu_open(driver, wait):
    # Tìm nút mở rộng
    expand_button = wait.until(EC.element_to_be_clickable((
        By.CSS_SELECTOR,"body > div.bg-\[\#F8F8F8\] > " \
        "div.relative.mx-auto.min-h-screen.w-1200px.overflow-x-auto.scrollbar-hide.pt-20px "
        "> div > div:nth-child(1) > div > div > button")))
    expand_button.click()
    time.sleep(1)


<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_30836\3549759051.py:4: SyntaxWarning: invalid escape sequence '\['
  By.CSS_SELECTOR,"body > div.bg-\[\#F8F8F8\] > " \


In [24]:
def crawl_medicine_links(driver, wait, category = None):

    if category is None:
        try:
            category = "Thuốc bổ và vitamin"
        except Exception:
            category = None

    print("Load tất cả sản phẩm")
    scroll_to_load_all(driver, wait)
    time.sleep(3)

    print("Đang thu thập liên kết thuốc không kê đơn...")    
    thuoc_khongkedon_list = driver.find_elements(By.CSS_SELECTOR, "a[href*='/thuoc'], a[href*='/dau']")
    time.sleep(2)
    
    print(f"Tìm thấy {len(thuoc_khongkedon_list)} elements")
    khongkedon_links_raw = [item.get_attribute("href") for item in thuoc_khongkedon_list]
    khongkedon_links = list(set(khongkedon_links_raw))
    print(f"Sau khi loại bỏ duplicate: {len(khongkedon_links)} links (đã loại {len(khongkedon_links_raw) - len(khongkedon_links)} duplicates)")
    time.sleep(2)

    # Kiểm tra xem có tab "Thuốc kê đơn" không - dùng find_elements thay vì find_element
    kedon_buttons = driver.find_elements(By.XPATH, "//button[contains(text(), 'Thuốc kê đơn')]")
    if not kedon_buttons:
        print("Không có tab 'Thuốc kê đơn', chỉ lấy thuốc không kê đơn")
        print(f"Tổng số liên kết thuốc thu thập được: {len(khongkedon_links)}")
        
        rows = [
            {"link": link, "category": category, "drug_type": "Không kê đơn"}
            for link in khongkedon_links
        ]
        df = pd.DataFrame(rows, columns =["link","category","drug_type"])
        return df
    
    print("Đang thu thập liên kết thuốc kê đơn...")
    kedon_button = kedon_buttons[0]
    time.sleep(1)
    driver.execute_script("arguments[0].click();", kedon_button)
    time.sleep(3)

    # Đóng pop-up
    close_popups(driver, wait)

    print("Load tất cả sản phẩm")
    scroll_to_load_all(driver, wait)
    time.sleep(3)
    thuoc_kedon_list = driver.find_elements(By.CSS_SELECTOR, "a[href*='/thuoc'], a[href*='/dau']")
    time.sleep(2)
    
    print(f"Tìm thấy {len(thuoc_kedon_list)} elements")
    kedon_links_raw = [item.get_attribute("href") for item in thuoc_kedon_list]
    kedon_links = list(set(kedon_links_raw))
    print(f"Sau khi loại bỏ duplicate: {len(kedon_links)} links (đã loại {len(kedon_links_raw) - len(kedon_links)} duplicates)")
    
    # Kiểm tra overlap giữa 2 tabs
    overlap = set(khongkedon_links) & set(kedon_links)
    if overlap:
        print(f"{len(overlap)} links xuất hiện ở cả 2 tabs!")
    
    total_links = khongkedon_links + kedon_links
    print(f"Tổng: {len(total_links)} links (Không kê đơn: {len(khongkedon_links)}, Kê đơn: {len(kedon_links)})")
    

    rows = [{"link": link, "category": category, "drug_type": "Không kê đơn"}
            for link in khongkedon_links
            ] + [
            {"link": link, "category": category, "drug_type": "Kê đơn"}
            for link in kedon_links]
    df = pd.DataFrame(rows, columns = ["link","category","drug_type"])
    return df

In [25]:
def crawl_medicine_menu_links(driver, wait):
    all_dfs = []
    
    category_menu_open(driver, wait)

    # Lấy số lượng danh mục
    categories = driver.find_elements(By.CSS_SELECTOR, "body > div.bg-\[\#F8F8F8\] > " \
    "div.relative.mx-auto.min-h-screen.w-1200px.overflow-x-auto.scrollbar-hide.pt-20px > " \
    "div > div:nth-child(1) > div > ul > li")
    count = len(categories)
    print(f"Tìm thấy {count} danh mục thuốc")
    
    for i in range(count):
        try:
            category_menu_open(driver, wait)
            
            # Tìm lại elements (tránh StaleElement)
            categories = wait.until(EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "body > div.bg-\[\#F8F8F8\] > "
                "div.relative.mx-auto.min-h-screen.w-1200px.overflow-x-auto.scrollbar-hide.pt-20px "
                "> div > div:nth-child(1) > div > ul > li")
            ))
            
            category = categories[i]
            
            # Cuộn đến category trong menu
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", category)
            time.sleep(0.5)
            
            name = category.text.strip()
            print(f"\n[{i+1}/{count}] Đang xử lý: {name}")
            
            # Click vào danh mục 
            driver.execute_script("arguments[0].click();", category)
            time.sleep(2)
            
            # Thu thập links thuốc
            df = crawl_medicine_links(driver, wait, category = name)
            all_dfs.append(df)
            print(f"Đã thu thập {len(df)} links từ '{name}'")
            
            # Quay lại trang trước
            driver.back()
            time.sleep(3)
            
        except Exception as e:
            print(f"Lỗi danh mục {i+1}: {e}")
            driver.back()
            time.sleep(2)
            continue
    
    if all_dfs:
        result_df = pd.concat(all_dfs, ignore_index= True)

        #result_df = result_df.drop_duplicates(subset=['link'])

        print(f"Tổng: {len(result_df)} links từ {len(all_dfs)} danh mục")    
        return result_df
    else:
        return pd.DataFrame()

<>:7: SyntaxWarning: invalid escape sequence '\['
<>:19: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\['
<>:19: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_30836\1555819899.py:7: SyntaxWarning: invalid escape sequence '\['
  categories = driver.find_elements(By.CSS_SELECTOR, "body > div.bg-\[\#F8F8F8\] > " \
C:\Users\Admin\AppData\Local\Temp\ipykernel_30836\1555819899.py:19: SyntaxWarning: invalid escape sequence '\['
  (By.CSS_SELECTOR, "body > div.bg-\[\#F8F8F8\] > "


In [26]:
def crawl_ankhang_medicine_links():
    driver = set_up_driver()
    wait = WebDriverWait(driver, 10)
    try: 
        print("Đang truy cập vào trang chủ An Khang...")
        driver.get("https://www.nhathuocankhang.com/")
        time.sleep(3)


        print("Đang truy cập vào trang danh mục thuốc...")
        thuoc_menu = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.z-10.w-\[232px\] > div > div.relative.z-\[2\].flex.text-black-100 > div.z-10.py-5.flex.flex-col.gap-4.min-h-\[320px\].h-full.w-full.bg-white.overflow-y-auto.rounded-16px > ul > li:nth-child(2)")))
        thuoc_menu.click()
        time.sleep(3)

        thuoc_menu_all = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > div > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.z-10.w-\[232px\] > div > div.relative.z-\[2\].flex.text-black-100 > div:nth-child(3) > div > div > div > a:nth-child(1)")))
        thuoc_menu_all.click()
        time.sleep(3)

        df_base = crawl_medicine_links(driver,wait)
       
        
        print("Thu thập danh sách danh mục thuốc")
        df_menu = crawl_medicine_menu_links(driver, wait)

        total_df = pd.concat([df_base, df_menu], ignore_index=True)
        #total_df = total_df.drop_duplicates(subset=['link'])
        print(f"Tổng số link thu thập được: {len(total_df)}")
        print(total_df.head)

        total_df.to_csv("ankhang_medicine_links.csv", index=False)
        return total_df
    
    except Exception as e:
        print(f"Lỗi: {e}")
        return []

    finally:
        driver.quit()

<>:11: SyntaxWarning: invalid escape sequence '\['
<>:15: SyntaxWarning: invalid escape sequence '\['
<>:11: SyntaxWarning: invalid escape sequence '\['
<>:15: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_30836\1881012351.py:11: SyntaxWarning: invalid escape sequence '\['
  thuoc_menu = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.z-10.w-\[232px\] > div > div.relative.z-\[2\].flex.text-black-100 > div.z-10.py-5.flex.flex-col.gap-4.min-h-\[320px\].h-full.w-full.bg-white.overflow-y-auto.rounded-16px > ul > li:nth-child(2)")))
C:\Users\Admin\AppData\Local\Temp\ipykernel_30836\1881012351.py:15: SyntaxWarning: invalid escape sequence '\['
  thuoc_menu_all = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > div > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.z-10.w-\[232px\] > div > div.relative.z-\[2\].flex.text-black-100 

In [27]:
if __name__ == "__main__":
    print ("Bắt đầu thu thập liên kết thuốc từ An Khang")
    print("="*50)
    crawl_ankhang_medicine_links()
    print("\n"+ "="*50)
    print("Hoàn tất thu thập liên kết thuốc từ An Khang")

Bắt đầu thu thập liên kết thuốc từ An Khang
Đang truy cập vào trang chủ An Khang...
Đang truy cập vào trang danh mục thuốc...
Load tất cả sản phẩm
Hoàn tất load: 329 sản phẩm
Đang thu thập liên kết thuốc không kê đơn...
Tìm thấy 329 elements
Sau khi loại bỏ duplicate: 82 links (đã loại 247 duplicates)
Đang thu thập liên kết thuốc kê đơn...
Load tất cả sản phẩm
Hoàn tất load: 1120 sản phẩm
Tìm thấy 1120 elements
Sau khi loại bỏ duplicate: 224 links (đã loại 896 duplicates)
Tổng: 306 links (Không kê đơn: 82, Kê đơn: 224)
Thu thập danh sách danh mục thuốc
Tìm thấy 15 danh mục thuốc

[1/15] Đang xử lý: Cơ xương khớp, gút
Load tất cả sản phẩm
Hoàn tất load: 48 sản phẩm
Đang thu thập liên kết thuốc không kê đơn...
Tìm thấy 48 elements
Sau khi loại bỏ duplicate: 12 links (đã loại 36 duplicates)
Đang thu thập liên kết thuốc kê đơn...
Load tất cả sản phẩm
Hoàn tất load: 255 sản phẩm
Tìm thấy 255 elements
Sau khi loại bỏ duplicate: 51 links (đã loại 204 duplicates)
Tổng: 63 links (Không kê đơn: 

In [30]:
ankhang_df = pd.read_csv("ankhang_medicine_links.csv")
print(ankhang_df.info())
print(ankhang_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2127 entries, 0 to 2126
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   link       2127 non-null   object
 1   category   2127 non-null   object
 2   drug_type  2127 non-null   object
dtypes: object(3)
memory usage: 50.0+ KB
None
                                                     link  \
count                                                2127   
unique                                               1969   
top     https://www.nhathuocankhang.com/thuoc-tri-ung-...   
freq                                                    2   

                   category drug_type  
count                  2127      2127  
unique                   15         2  
top     Thuốc bổ và vitamin    Kê đơn  
freq                    447      1576  


In [31]:
catgory_count = ankhang_df['category'].value_counts()
print(catgory_count)

category
Thuốc bổ và vitamin                            447
Kháng sinh, kháng nấm                          259
Giảm đau, hạ sốt, kháng viêm                   218
Thần kinh, não bộ                              216
Tim mạch, tiểu đường, mỡ máu                   205
Da liễu, dị ứng                                203
Tiêu hóa, gan mật                              199
Hô hấp                                         148
Tiết niệu, sinh dục                            106
Cơ xương khớp, gút                              63
Dầu, Cao Xoa, Miếng Dán                         23
Mắt, tai mũi họng                               23
Thuốc điều trị ung thư, miễn dịch               10
Thuốc làm đẹp, giảm cân                          5
Thuốc giải độc, khử độc - hỗ trợ cai nghiện      2
Name: count, dtype: int64
